In [39]:
import triton
import triton.language as tl

@triton.jit

def fused_add_relu_kernel(
    x_ptr,
    bias_ptr,
    y_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
  pid=tl.program_id(0)

  block_start=pid*BLOCK_SIZE

  offsets=block_start+tl.arange(0, BLOCK_SIZE)

  mask=offsets<n_elements

  x=tl.load(x_ptr+offsets, mask=mask)
  b=tl.load(bias_ptr+offsets, mask=mask)

  y=x+b
  y=tl.maximum(y, 0.0)

  tl.store(y_ptr+offsets, y, mask=mask)




In [40]:
import torch
def fused_add_relu(x, bias):
  y=torch.empty_like(x)
  n_elements=x.numel()

  grid=lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)

  fused_add_relu_kernel[grid](
      x,
      bias,
      y,
      n_elements,
      BLOCK_SIZE=1024,
  )
  return y


In [41]:
x = torch.randn(10_000_000, device="cuda")
bias = torch.randn(10_000_000, device="cuda")

y_torch = torch.relu(x + bias)
y_triton = fused_add_relu(x, bias)

torch.cuda.synchronize()

print(torch.allclose(y_torch, y_triton))

True


In [42]:
import time

torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(100):
    y = torch.relu(x + bias)

torch.cuda.synchronize()
end = time.perf_counter()

print("PyTorch:", (end - start) / 100)

PyTorch: 0.00083287073000065


In [43]:
torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(100):
    fused_add_relu_kernel[grid](
        x,
        bias,
        y_triton,
        x.numel(),
        BLOCK_SIZE=1024,
    )

torch.cuda.synchronize()
end = time.perf_counter()

print("Triton:", (end - start) / 100)

Triton: 3.2346209998195264e-05
